# Sliding Window

A pattern for problems over **contiguous** subarrays or substrings: instead of re-examining every window from scratch (O(n·k) or O(n²)), keep a window of elements and **slide** it — adding what enters, removing what leaves — for a single **O(n)** pass.

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **Fixed-size windows**
3. **Variable-size windows** — grow & shrink
4. **When to use & toolkit**
5. **Complexity cheat-sheet**

## 1. The signal — when to reach for a sliding window

Reach for it when the problem is about a **contiguous run** — a subarray or substring — and asks for an optimum or a count over all such runs:

- "longest / shortest / max-sum / min subarray or substring such that …"
- the brute force is "try every window," which is O(n·k) for a fixed width k, or O(n²) for a variable one.

The trick: adjacent windows overlap almost entirely, so recomputing each from scratch is wasteful. Keep a running summary (a sum, a count, a character set) and **update it incrementally** as the window moves — O(1) per step, O(n) total. Two flavors: a **fixed** width, or a **variable** width that grows and shrinks to satisfy a condition.

## 2. Fixed-size windows

When the width k is given, slide one step at a time: **add the entering element and subtract the leaving one.** No inner loop — each element is added once and removed once, so it's O(n) instead of the O(n·k) "sum every window" approach.

In [1]:
def max_sum_subarray(a, k):
    window = sum(a[:k])              # the first window
    best = window
    for i in range(k, len(a)):
        window += a[i] - a[i - k]    # add the new element, drop the one that left
        best = max(best, window)
    return best

print("max_sum_subarray([2,1,5,1,3,2], 3):", max_sum_subarray([2, 1, 5, 1, 3, 2], 3))


max_sum_subarray([2,1,5,1,3,2], 3): 9


Tracking the *maximum* (rather than the sum) of each fixed window needs a bit more — a **monotonic deque** keeps it O(n). That's the monotonic-stack-and-queue notebook.

## 3. Variable-size windows — grow & shrink

When the width isn't fixed, use **two indices** for the window's ends. Expand `right` to grow the window; when it violates the condition (or once it meets a target), advance `left` to shrink it. Each index only ever moves **forward**, so even with the inner shrink loop the total work is **O(n) amortized** — every element enters and leaves the window at most once.

The first template **maximizes** a window that must stay valid — the longest substring with no repeated character. Track each character's last position, and when a repeat falls inside the window, jump `left` past it:

In [2]:
def longest_unique(s):
    last = {}                        # char -> its most recent index
    left = best = 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1      # shrink: skip past the earlier duplicate
        last[ch] = right
        best = max(best, right - left + 1)
    return best

for s in ("abcabcbb", "bbbbb", "pwwkew"):
    print(f"longest_unique({s!r}):", longest_unique(s))


longest_unique('abcabcbb'): 3
longest_unique('bbbbb'): 1
longest_unique('pwwkew'): 3


The mirror template **minimizes** a window that must *reach* a target — the smallest subarray whose sum is at least `target`. Grow until the sum qualifies, then shrink from the left as far as it still qualifies, recording the smallest width seen:

In [3]:
def min_subarray_len(target, a):
    left = total = 0
    best = float("inf")
    for right, x in enumerate(a):
        total += x                   # grow the window
        while total >= target:       # shrink while it still satisfies the target
            best = min(best, right - left + 1)
            total -= a[left]
            left += 1
    return best if best != float("inf") else 0

print("min_subarray_len(7, [2,3,1,2,4,3]):", min_subarray_len(7, [2, 3, 1, 2, 4, 3]))
print("min_subarray_len(15, [1,2,3,4])   :", min_subarray_len(15, [1, 2, 3, 4]), "(impossible -> 0)")


min_subarray_len(7, [2,3,1,2,4,3]): 2
min_subarray_len(15, [1,2,3,4])   : 0 (impossible -> 0)


## 4. When to use & toolkit

| The problem says… | Window |
|---|---|
| "subarray / substring of size k …" | fixed |
| "longest / shortest … such that *condition*" | variable (grow/shrink) |
| "max in every window of size k" | fixed + a monotonic deque |
| "count of windows where …" | variable; add `right - left + 1` per step |

**Python toolkit:** `collections.deque` holds a window's elements (or a monotonic max/min) with O(1) ends; `collections.Counter` tracks element frequencies inside the window (the engine for "permutation in string" / "minimum window substring"); for range queries that *aren't* a moving window, reach for the prefix-sums notebook.

**Watch for:** the metric must be **incrementally updatable**, and shrinking on a *sum* target assumes **non-negative** values (with negatives a window sum isn't monotonic — use prefix sums + a hashmap instead).

## 5. Complexity cheat-sheet

| Variant | Brute force | Sliding window |
|---|:---:|:---:|
| Fixed size k | O(n·k) | O(n) |
| Variable (grow / shrink) | O(n²) | O(n) amortized |
| Max / min per fixed window | O(n·k) | O(n) with a monotonic deque |

<sub>Every element enters and leaves the window at most once — which is why even the variable form, inner shrink loop and all, stays linear.</sub>